# 02 — Train a DQN Model Offline

Load a stored transition dataset and train a MOUSE model entirely from replayed data, with no live environment in the training loop.

A MOUSE model is assembled from three independent pieces:

- `StepEmbedder` converts each `(action, observation, reward, done)` step into a token sequence the backbone can process.
- `Qwen3Backbone` reads those tokens with a transformer and builds up context over the sequence.
- `DiscreteActionValueHead` maps the backbone's output to one Q-value per action.

`Qwen3Backbone` downloads the full `Qwen/Qwen3-0.6B` backbone from the Hub on first run. Its `hidden_dim` (1024 for this checkpoint) is the single number that connects all three pieces — no manual dimension matching required. For quick debugging you can pass `num_layers=...` to truncate the backbone, but this tuned training example uses all pretrained layers.

In [1]:
import torch

from mouse_core.data import (
    DataLoader,
    Augmenter,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"   # Hub repo id for load_stores_from_hub
MODEL_ID = "mouse-example-model-offline"       # Hub repo id for push_model_to_hub
MAX_ACTIONS = 4                        # discrete action head size (FrozenLake: up/down/left/right)
MAX_OBS_DISCRETE = 64                  # observation embedder vocab (max cells in 8×8 maps)
GRADIENT_STEPS = 20000                 # total optimizer updates (same budget as 03_train_online)
SEQUENCE_LENGTH = 512                  # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                         # sequences per optimizer step


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load data

`load_stores_from_hub` downloads the matching dataset parquet shards as a local Hub snapshot, then reconstructs the `Datastore` objects — one per stored maze stream.

In [ ]:
stores = load_stores_from_hub(DATASET_ID, split="train", force_download=True)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 50 files:   0%|          | 0/50 [00:00<?, ?it/s]

Stores 10 of 50:
Datastore(name='proc_frozenlake_0', steps=30000, columns=['action', 'time', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_q_star'])
Datastore(name='proc_frozenlake_1', steps=30000, columns=['action', 'time', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_q_star'])
Datastore(name='proc_frozenlake_10', steps=30000, columns=['action', 'time', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_q_star'])
Datastore(name='proc_frozenlake_11', steps=30000, columns=['action', 'time', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_q_star'])
Datastore(name='proc_frozenlake_12', steps=30000, columns=['action', 'time', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_q_star'])
Datastore(name='proc_frozenlake_13', steps=30000, columns=['action', 'time', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_q_star'])
Datastore(name='proc_frozenlake_14', steps=

## DataLoader and augmentation

`DataLoader` slices one or more `Datastore`s into fixed-length sequence batches and mixes them automatically — no manual merging required.

- **`sequence_length=512`** — each item in a batch is 512 consecutive steps from a single env instance.
- **`batch_size=4`** — 4 sequences per call to `next_batch()`.
- **`prefetch=4`** — pre-loads 4 batches in the background to keep the GPU fed.
- **`seed`** — optional; defaults to `None` for non-deterministic sampling. Pass an integer for reproducible batches.

`next_batch()` returns a `list[list[dict]]` of shape `[batch][sequence]` — one inner list per sequence, one dict per step. Passing `augmenter=augmenter` makes `DataLoader` run `Augmenter` inside the sampling path. Augmentation modality specs use `field` plus an augmentation `type` such as `discrete`, `linear`, or `image`. The tuned baseline defines the augmenter but leaves it disabled in the loader.

In [3]:
augmenter = Augmenter(
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
    # is_seam is the pack-mode segment marker DqnObjective uses to skip
    # fake transitions across packed-segment boundaries.
    keep_fields=["action", "observation", "reward", "done", "is_seam"],
)

loader = DataLoader(
    stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    prefetch=4,
    num_workers=0,
    pack=True,
    augmenter=augmenter,
)

## Build the model

All three components are sized by `backbone.hidden_dim`, so they connect without any manual dimension matching.

**`StepEmbedder`** — each entry in `modalities` describes one field from the step dict and which `type` to use:
- `"discrete"` — looks up a learned vector from a small vocabulary. Use for integer-valued fields like actions, observations, and done codes.
- `"rff"` — embeds a continuous scalar using random Fourier features. Use for rewards or any float value.
- `"learnable"` — a free learnable token with no input, giving the model scratch space it can write to without consuming an observation slot.

`include_type_token=False` disables the per-modality type embedding that would otherwise be added on top of every content embedding. With all modalities at the same `std=0.02` the type signal is already balanced, but disabling it keeps the summed token purely content-driven.

**`DiscreteActionValueHead`** — a small MLP that predicts one Q-value per action. The tuned baseline uses `scale=1.0` for the action-value head.

**`Model`** — wraps encoder, backbone, and head(s) into a single object with a unified `forward` call.

In [4]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1
        },
    ],
    modality_fusion="sum",
    include_type_token=False,
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
model.backbone.model.compile()  # one-time warmup; speeds up training forwards (cached decode uses FlexAttention internally)
print(model)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model(
  (encoder): StepEmbedder(
    (type_embedder): TypeEmbedder(
      (embed): ScaledEmbedding(64, 1024)
    )
    (modality_embedders): ModuleDict(
      (action): DiscreteEmbedder(
        (embed): ScaledEmbedding(4, 1024)
      )
      (observation): DiscreteEmbedder(
        (embed): ScaledEmbedding(64, 1024)
      )
      (reward): ScalarRFFEmbedder(
        (rff): RandomFourierFeatures()
      )
      (done): DiscreteEmbedder(
        (embed): ScaledEmbedding(5, 1024)
      )
    )
  )
  (backbone): Qwen3Backbone(
    (model): Qwen3Model(
      (embed_tokens): Embedding(1, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=10

## Train

Each iteration:
1. Sample an already-augmented batch of transition sequences from the data loader.
2. Run both the online network and a frozen target network to get Q-value predictions.
3. Compute the Bellman loss — `objective(objective_data, predictions)` returns `(loss, metrics)`.
4. Back-propagate, clip gradients, and step the optimizer.
5. Slowly copy online weights → target network (Polyak/EMA update) for stable TD targets.

### Discount factors and done codes

MOUSE uses five `done` codes. `DqnObjective` maps each to its own bootstrap discount:

- `0`: running, using `gamma_step`.
- `1`: episode done within a task, using `gamma_episode_terminal`.
- `2`: episode truncated within a task, using `gamma_episode_truncated`.
- `3`: task done, using `gamma_task_terminal`.
- `4`: task truncated, using `gamma_task_truncated`.

The example dataset is collected with `episodes_per_task=5` and a fresh map per task, and the gammas mirror that structure exactly:

- **Within a task everything is undiscounted** (`gamma_step = gamma_episode_* = 1.0`). Q-values mean one thing: *expected remaining points in this task* (0 to 5). Dying in a hole and timing out are identical — each costs exactly one of the task's 5 episodes and nothing else, because no value decays with time. There is no wormhole: reaching future episodes sooner gains nothing when nothing is discounted.
- **Task boundaries are a hard cut** (`gamma_task_* = 0.0`). The next task is a different map with a different labeling; its value is not this task's business. This matches inference, where the KV-cache is reset at task boundaries.

The within-episode progress pressure comes from the environment, not the discount: with `max_episode_steps=30`, a wasted step matters exactly when it makes the goal unreachable before the deadline, forfeiting that episode's `+1`. That signal is sparse but large (a full point at the critical step, versus the fraction-of-a-percent tick a discount would provide), and the model must infer time-to-deadline from context — steps are not individually penalized and time is not an input.

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8
)
objective = DqnObjective(
    gamma_step=1.0,                 # undiscounted within a task: value = expected remaining points
    gamma_episode_terminal=1.0,     # value flows freely across episode boundaries in a task
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,        # hard cut at task boundaries: each task is its own problem
    gamma_task_truncated=0.0,
    tau=0.0005,
)

for step in range(GRADIENT_STEPS):

    batch = loader.next_batch()

    predictions, objective_data, _ = model(batch)
    loss, metrics = objective(objective_data, predictions)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.polyak_update(action_value_tau=objective.tau)

    if step % 100 == 0:
        print(f"step={step}  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")

loader.close()

/home/user/Repos/mouse-core/.venv/lib/python3.13/site-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


step=0  loss=0.0762  q=-0.004
step=100  loss=0.0515  q=0.772
step=200  loss=0.0186  q=0.608
step=300  loss=0.0186  q=0.483
step=400  loss=0.0161  q=0.653
step=500  loss=0.0226  q=0.687
step=600  loss=0.0132  q=0.574
step=700  loss=0.0090  q=0.514
step=800  loss=0.0156  q=0.591
step=900  loss=0.0298  q=0.462
step=1000  loss=0.0125  q=0.860
step=1100  loss=0.0131  q=0.734
step=1200  loss=0.0127  q=0.705
step=1300  loss=0.0126  q=0.824
step=1400  loss=0.0097  q=0.684
step=1500  loss=0.0112  q=0.702
step=1600  loss=0.0122  q=0.732
step=1700  loss=0.0101  q=0.612
step=1800  loss=0.0114  q=0.687
step=1900  loss=0.0117  q=0.772
step=2000  loss=0.0113  q=0.893
step=2100  loss=0.0100  q=0.757
step=2200  loss=0.0165  q=0.726
step=2300  loss=0.0103  q=0.850
step=2400  loss=0.0103  q=0.670
step=2500  loss=0.0124  q=0.799
step=2600  loss=0.0130  q=0.891
step=2700  loss=0.0107  q=0.952
step=2800  loss=0.0116  q=0.820
step=2900  loss=0.0081  q=0.972
step=3000  loss=0.0159  q=0.933
step=3100  loss=0.0

## Push to the Hub

Save the offline-trained model to the shared Hub repo under `MODEL_ID`. The inference notebook loads the current checkpoint from that repo, so whichever training notebook you push last is the model it evaluates.

In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")